# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
from jax.typing import ArrayLike
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from functools import partial
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    run_worm
)
import os
import jax
n_cpus = os.cpu_count()
n_used_cpus = n_cpus
print("Number of CPUs available: {}".format(n_cpus))
print("Number of used CPUs: {}".format(n_used_cpus))
jax.config.update('jax_num_cpu_devices', n_used_cpus)
jax.config.update("jax_log_compiles", True)

Number of CPUs available: 16
Number of used CPUs: 16


In [2]:
jax.devices()

[CpuDevice(id=0),
 CpuDevice(id=1),
 CpuDevice(id=2),
 CpuDevice(id=3),
 CpuDevice(id=4),
 CpuDevice(id=5),
 CpuDevice(id=6),
 CpuDevice(id=7),
 CpuDevice(id=8),
 CpuDevice(id=9),
 CpuDevice(id=10),
 CpuDevice(id=11),
 CpuDevice(id=12),
 CpuDevice(id=13),
 CpuDevice(id=14),
 CpuDevice(id=15)]

In [3]:
n_cpus = os.cpu_count()
print(n_cpus)

16


In [31]:
length = 7  # 5
width = 7  # 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_z_mod_2 = moebius_code.h_z_mod_2
h_z_mod_p = moebius_code.h_z_mod_p
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
num_plaquette, num_edges = h_x.shape
gamma_t = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

# error_master_key = jax.random.PRNGKey(42)
# initial_error = em_lindblad.generate_random_error(error_key)
# initial_error_mod_2 = jnp.mod(initial_error, 2)
# initial_error_mod_p = jnp.mod(initial_error, p)
# # Here we consider the full syndrome including the plaquette
# # we usually remove because of the constraint as this simplified the
# # coding of the worm algorithm. In fact, in this the syndromes will
# # always be annihilated in pairs, and the total number of syndromes is
# # always even as one can check numerically.
# syndrome = moebius_code.get_plaquette_syndrome(initial_error)
# syndrome_mod_2 = jnp.mod(syndrome, 2)
# syndrome_mod_p = jnp.mod(syndrome, p)
# num_plaquette, num_edges = h_x.shape

In [32]:
master_worm_key = jax.random.PRNGKey(144)
num_worms = 10 * n_cpus
worm_keys = jax.random.split(master_worm_key, num_worms)

error_master_key = jax.random.PRNGKey(42)
num_samples = 3 * 10 * n_cpus
error_keys = jax.random.split(error_master_key, num_samples)


def generate_initial_worm_errors(
    key: ArrayLike,
    error_model: ErrorModelLindbladTwoOddPrime
) -> ArrayLike:
    initial_error = error_model.generate_random_error(key)
    initial_error_mod_2 = jnp.mod(initial_error, 2)
    initial_error_mod_p = jnp.mod(initial_error, p)
    initial_worm_error = jnp.vstack((initial_error_mod_2, initial_error_mod_p))
    return initial_worm_error


initial_worm_errors = jax.vmap(
    generate_initial_worm_errors, in_axes=(0, None))(error_keys, em_lindblad)

In [33]:
# Sharding the arrays
devices = jax.devices()  # Assuming this returns 16 devices
mesh = Mesh(devices, ('batch',))

sharding_for_keys = NamedSharding(mesh,  PartitionSpec('batch', None))
worm_keys_sharded = jax.device_put(worm_keys, sharding_for_keys)

# 2. Define sharding: Split the 0th axis across 'batch', leave others whole
sharding_for_error = NamedSharding(mesh,  PartitionSpec('batch', None, None))
initial_worm_errors_sharded = jax.device_put(
    initial_worm_errors, sharding_for_error)

In [34]:
jax.debug.visualize_array_sharding(initial_worm_errors_sharded[:, 1, :])

  CPU 0  
  CPU 1  
  CPU 2  
  CPU 3  
  CPU 4  
  CPU 5  
  CPU 6  
  CPU 7  
  CPU 8  
  CPU 9  
 CPU 10  
 CPU 11  
 CPU 12  
 CPU 13  
 CPU 14  
 CPU 15  

In [35]:
jax.debug.visualize_array_sharding(worm_keys_sharded)

  CPU 0  
  CPU 1  
  CPU 2  
  CPU 3  
  CPU 4  
  CPU 5  
  CPU 6  
  CPU 7  
  CPU 8  
  CPU 9  
 CPU 10  
 CPU 11  
 CPU 12  
 CPU 13  
 CPU 14  
 CPU 15  

In [36]:
# master_worm_key = jax.random.PRNGKey(144)
# num_worms = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)

# error_master_key = jax.random.PRNGKey(42)
# num_samples = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)
# initial_error = em_lindblad.generate_random_error(error_key)

max_steps = 1000
initial_worm_state = {}
# worm_error = jnp.vstack(
#     (initial_error_mod_2, initial_error_mod_p))
initial_head = 18
initial_worm_state["head"] = initial_head
initial_worm_state["tail"] = initial_head
initial_worm_state["worm_success"] = False
initial_worm_state["accepted_moves"] = 0
initial_worm_state["attempted_moves"] = 0

run_worm_partial = partial(
    run_worm,
    initial_worm_state=initial_worm_state,
    h_error_mod_p=h_z_mod_p,
    h_mod_p=h_x_mod_p,
    error_model=em_lindblad,
    max_worm_steps=max_steps
)

# First over keys
run_worm_vmap = jax.vmap(run_worm_partial, in_axes=(None, 0))
# Then over initial errors
run_worm_vmap = jax.vmap(run_worm_vmap, in_axes=(0, None))
run_worm_jit = jax.jit(run_worm_vmap)
new_worm_state = run_worm_jit(initial_worm_errors_sharded, worm_keys)

# index = 0
# print("Number of accepted moves: {}".format(new_worm_state["accepted_moves"][index]))
# print("Number of attempted moves: {}".format(
#     new_worm_state["attempted_moves"][index]))

In [37]:
index = 28
print("Number of accepted moves: {}".format(
    new_worm_state["accepted_moves"][index]))
print("Number of attempted moves: {}".format(
    new_worm_state["attempted_moves"][index]))
print("Success: {}".format(
    new_worm_state["worm_success"][index]))

Number of accepted moves: [ 4  4  4  2  4  2  6  2  2  2  2  2  2  2  4  2  2  2  2  6  2  2  8  2
  2  6 10  4  4  2  2  2  2  2  2  2  2  2  2  2  2  4  2  2  4  2 14  2
  2  2  2  2  2  2  2  2  2  2  2  6  2  2  2  2  2  8  2  2  2  2  4  2
  2  2  4  2  2  6  2  2  2  2  4  2  2  2  4  2  2  2  2  2  4  2  2  4
  2  2  4  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  4  2  2
  4  8  2  2  4  6  2  2  2  2 10  2  2  2  2  6  2  2  2  2 22  2  2 10
  2  2  2  2  2  2  4  2  2 20  2  2  2  2  2  2]
Number of attempted moves: [ 99  72 180  67 215 294 143 144  99  40  24  75  89 218  43 204  30 126
   8 148  68  57 109  16  41  91 118  24 182  11 105 116  32 100 220  52
  33  98  22  42  35  24 688  31  63  53 193  31  18  58 133 112  12  55
  14  94  15 110  91  23 127  15  30 178 156 160  55 103 103  12  33  58
  39  48 123  20  60  61  56  10 351  33 291  10  44  46  88 124  10  28
  62  17  31 121  97  71 255  51  49 198  10 171  51   8 152  56 165  36
  85 110  84  27  40 

In [19]:
print(new_worm_state["head"])
print(initial_worm_state["head"])

[[18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 ...
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]]
18


In [20]:
initial_worm_errors[0][0]

Array([0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0], dtype=int32)

In [21]:
index_a = 0
index_b = 0
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][0, :] - initial_worm_errors[index_a][0, :], 2))))

1


In [22]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][1, :] - initial_worm_errors[index_a][1, :], p))))

2


In [23]:
syndrome_mod_2 = jnp.mod(h_x_mod_2 @ initial_worm_errors[index_a][0, :], 2)
new_syndrome_mod_2 = jnp.mod(
    h_x_mod_2 @ new_worm_state["worm_error"][index_a, index_b][0, :], 2)

jnp.mod(new_syndrome_mod_2 - syndrome_mod_2, 2)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [24]:
syndrome_mod_p = jnp.mod(h_x_mod_p @ initial_worm_errors[index_a][1, :], p)
new_syndrome_mod_p = jnp.mod(
    h_x_mod_p @ new_worm_state["worm_error"][index_a, index_b][1, :], p)

jnp.mod(new_syndrome_mod_p - syndrome_mod_p, p)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)